In [1]:
%pip install HolisticTraceAnalysis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 182.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 238.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 218.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55/55 [HolisticTraceAnalysis]Analysis]r]nt]
Note: you may need to restart the kernel to use updated packages.


In [14]:
from hta.trace_analysis import TraceAnalysis

In [15]:
analyzer = TraceAnalysis(trace_dir = "../")

If the trace file does not have the rank specified in it, then add the following snippet key to the json files to use HTA; "distributedInfo": {"rank": 0}. If there are multiple traces files, then each file should have a unique rank value.For now we will default to rank = 0.
Parsed ../train-gpt-0-chrome-trace.json.gz time = 13.41 seconds 
Rounding down ns resolution events due to issue with events overlapping. ts dtype = float64, dur dtype = float64.Please see https://github.com/pytorch/pytorch/pull/122425
Parsed ../train-gpt-0-chrome-trace.json.gz backend=ParserBackend.JSON in 47.87 seconds; current PID:23734
Overall parsing of ../train-gpt-0-chrome-trace.json.gz in 60.90 seconds; current PID:23734
leaving parse_multiple_ranks duration=62.50 seconds
leaving parse_traces duration=62.51 seconds
ProfilerStep not found in the trace. The analysis result may not be accurate.


In [16]:
# Temporal breakdown
analyzer.get_temporal_breakdown()

,rank,idle_time(us),compute_time(us),non_compute_time(us),kernel_time(us),idle_time_pctg,compute_time_pctg,non_compute_time_pctg
0,0,1461641.0,7290908.0,17262.0,8769811.0,16.67,83.14,0.2


In [17]:
# Kernel breakdown
analyzer.get_gpu_kernel_breakdown()

(                      kernel_type      sum  percentage
 0                     COMPUTATION  7288698        99.7
 1                          MEMORY    19839         0.3
 2  COMPUTATION overlapping MEMORY     1839         0.0,
                                                  name   sum (us)   max (us)  \
 0     nvjet_sm90_tst_128x128_64x6_2x1_v_bz_splitK_NTT   918352.0   918352.0   
 1      nvjet_sm90_tst_256x128_64x4_1x2_h_bz_coopA_NNT  1003965.0  1003965.0   
 2      nvjet_sm90_tst_256x128_64x4_1x2_h_bz_coopA_TNT   999696.0   999696.0   
 3                                              others  2534035.0   167606.0   
 4   triton_per_fused__fused_rms_norm__fused_rms_no...   211263.0   211263.0   
 5   triton_per_fused__fused_rms_norm__to_copy__uns...   262281.0   262281.0   
 6   triton_per_fused__fused_rms_norm_backward__to_...   468301.0   468301.0   
 7   triton_per_fused__fused_rms_norm_backward__to_...   170713.0   170713.0   
 8                          triton_red_fused_mul_sum_8 

In [18]:
# Idle time breakdown
analyzer.get_idle_time_breakdown()

(   rank stream idle_category  idle_time  idle_time_ratio
 0     0      7     host_wait  1265964.0             0.76
 1     0      7   kernel_wait   395670.0             0.24
 2     0      7         other       68.0             0.00,
 None)

In [19]:
# Communication computation overlap
analyzer.get_comm_comp_overlap()

/home/fabrizio/workspace/parameter-golf/.venv/lib/python3.12/site-packages/hta/analyzers/communication_analysis.py:73: RuntimeWarning: invalid value encountered in scalar divide
  return (shifted_overlap["time_y"] - shifted_overlap["time_x"]).sum() / (


,rank,comp_comm_overlap_pctg
0,0,NaN


In [20]:
# Frequent CUDA kernel patterns
# analyzer.get_frequent_cuda_kernel_patterns(operator_name="aten::linear", output_dir="/new/trace/path")

In [21]:
# CUDA kernel launch statistics
analyzer.get_cuda_kernel_launch_stats()

{0:        correlation  cpu_duration  gpu_duration  launch_delay
 0               27          40.0           0.0           0.0
 1               46          49.0           6.0           1.0
 2               62          19.0           6.0           3.0
 3              114          10.0           0.0           3.0
 4              139           2.0          -1.0          31.0
 ...            ...           ...           ...           ...
 94743      1245001           4.0         622.0        2007.0
 94744      1245006           4.0          63.0        2624.0
 94745      1245020           2.0           0.0        2622.0
 94746      1245045           2.0           0.0        2884.0
 94747      1245058           1.0           0.0        2956.0
 
 [94748 rows x 4 columns]}

In [22]:
# Memory bandwidth time series
analyzer.get_memory_bw_time_series()

{0:               ts pid         name  memory_bw_gbps
 120     145295.0   0  Memcpy DtoD     1074.360656
 42120   145296.0   0  Memcpy DtoD        0.000000
 121     145344.0   0  Memcpy DtoD     1236.528302
 42121   145345.0   0  Memcpy DtoD        0.000000
 122     145437.0   0  Memcpy DtoD     1092.266667
 ...          ...  ..          ...             ...
 69999  8814889.0   0       Memset        0.000000
 28000  8814902.0   0       Memset        0.031281
 70000  8814903.0   0       Memset        0.000000
 28001  8814917.0   0       Memset        0.031250
 70001  8814918.0   0       Memset        0.000000
 
 [84000 rows x 4 columns]}

In [23]:
# Memory bandwidth summary
analyzer.get_memory_bw_summary()

memory_bw_gbps                                       \
                          count         mean        std          min   
rank name                                                              
0    Memcpy DtoD        13200.0  1172.624276  54.018478  1008.246154   
     Memcpy HtoD         1600.0    36.584899   0.956858    25.359776   
     Memset             24233.0     0.011401   0.012929     0.001147   

                                                                      
                          25%          50%          75%          max  
rank name                                                             
0    Memcpy DtoD  1130.540162  1170.285714  1191.563636  1365.333333  
     Memcpy HtoD    36.735426    36.900901    36.900901    37.236364  
     Memset          0.003906     0.005000     0.005435     0.047548

In [24]:
# Queue length time series
analyzer.get_queue_length_time_series()

{0:               ts pid tid  stream  queue_length
 index                                         
 3382604    60289   0   7       7             1
 3382602    60327   0   7       7             0
 3382610    60815   0   7       7             1
 3382608    60865   0   7       7             0
 3382616    60887   0   7       7             1
 ...          ...  ..  ..     ...           ...
 3832573  8829822   0   7       7             0
 3832581  8830094   0   7       7             1
 3832579  8830103   0   7       7             0
 3832585  8830128   0   7       7             1
 3832583  8830136   0   7       7             0
 
 [319096 rows x 5 columns]}

In [25]:
# Queue length summary
analyzer.get_queue_length_summary()

queue_length                                                   
                   count       mean        std  min  25%   50%   75%    max
rank stream                                                                
0    7          319096.0  35.820625  30.838447  0.0  9.0  31.0  56.0  201.0